# Partner Effectiveness Analysis: Driving Safehouse Outcomes
## v2 — Improved with Panel Data, Role Types, Diagnostics & Cross-Validation

### 1. Problem Framing

**Business Problem:** Lighthouse Sanctuary contracts with individuals and organizations to provide services to their safehouses. Leadership needs to know which partner characteristics—type, role, experience, workload—are associated with better resident outcomes, so they can make data-driven contract renewal and hiring decisions.

**Modeling Goal: Explanatory.** We want to understand *which partner factors* drive outcomes, not just predict a future score. Per the textbook (Ch. 9–10), this means we prioritize interpretable coefficients, check OLS assumptions, and control for confounders before drawing conclusions.

**Improvements over v1:**
| Issue in v1 | Fix in v2 |
|---|---|
| Only 21 rows (safehouse-level aggregation) | Panel dataset: 250 monthly observations |
| `role_type` column never used | `role_type` included as a key feature |
| Only `education_progress` modeled | Both `education_progress` and `health_score` modeled |
| No cross-validation | Leave-One-Out CV + Repeated K-Fold (Ch. 15) |
| No OLS assumption checks | Full diagnostic suite: VIF, residuals, Durbin-Watson, linearity (Ch. 10) |
| Activity metrics ignored | `process_recording_count` + `home_visitation_count` as control variables |
| In-sample tree with no evaluation | CV-compared MLR vs. Decision Tree |

## 2. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import probplot
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, LeaveOneOut, RepeatedKFold
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titlesize'] = 13

# ── Update BASE_PATH to your CSV folder ──────────────────────────────
BASE_PATH = 'lighthouse_csv_v7/'

partners    = pd.read_csv(BASE_PATH + 'partners.csv',               parse_dates=['start_date'])
assignments = pd.read_csv(BASE_PATH + 'partner_assignments.csv',    parse_dates=['assignment_start','assignment_end'])
metrics     = pd.read_csv(BASE_PATH + 'safehouse_monthly_metrics.csv', parse_dates=['month_start','month_end'])
safehouses  = pd.read_csv(BASE_PATH + 'safehouses.csv')

print(f'partners: {partners.shape} | assignments: {assignments.shape}')
print(f'metrics:  {metrics.shape}  | safehouses:  {safehouses.shape}')

## 3. Data Preparation — Panel Dataset

**Why panel instead of safehouse-level aggregates?**  
v1 aggregated all monthly metrics into a single mean per safehouse, leaving only 21 rows — far too few for reliable regression. Here we build a *panel*: for each monthly metric row, we identify which primary partner was active that month (using assignment date ranges), giving us ~250 observations with full time-variation.

We also bring in `role_type` (the specific function a partner performs), which was present in `partners.csv` but missing from v1, and add `process_recording_count` and `home_visitation_count` as activity control variables.

In [ ]:
# ── Partner feature engineering ───────────────────────────────────────
analysis_date = pd.Timestamp('2024-12-31')
partners['tenure_days'] = (analysis_date - partners['start_date']).dt.days

partner_workload = assignments.groupby('partner_id').size().reset_index(name='safehouse_count')
partners = partners.merge(partner_workload, on='partner_id', how='left').fillna({'safehouse_count': 0})

# ── Link each metric-month to its active primary partner ──────────────
primary = assignments[assignments['is_primary'] == True].copy()
primary['assignment_end_filled'] = primary['assignment_end'].fillna(pd.Timestamp('2099-12-31'))

rows = []
for _, m in metrics.iterrows():
    sh, mo = m['safehouse_id'], m['month_start']
    match = primary[
        (primary['safehouse_id'] == sh) &
        (primary['assignment_start'] <= mo) &
        (primary['assignment_end_filled'] >= mo)
    ]
    if len(match) > 0:
        row = m.to_dict()
        row['partner_id']   = match.iloc[0]['partner_id']
        row['program_area'] = match.iloc[0]['program_area']
        rows.append(row)

panel = pd.DataFrame(rows)

# ── Merge partner and safehouse attributes ────────────────────────────
panel = panel.merge(
    partners[['partner_id','partner_type','role_type','tenure_days','safehouse_count']],
    on='partner_id', how='left'
)

sh_info = safehouses[['safehouse_id','capacity_girls','current_occupancy','region']].copy()
sh_info['occupancy_rate'] = sh_info['current_occupancy'] / sh_info['capacity_girls']
panel = panel.merge(sh_info, on='safehouse_id', how='left')

# ── Time control: months since dataset start ──────────────────────────
panel['month_num'] = (
    (panel['month_start'].dt.year - 2023) * 12 + panel['month_start'].dt.month
)

# ── Drop rows missing either outcome ─────────────────────────────────
panel_clean = panel.dropna(subset=['avg_education_progress', 'avg_health_score']).copy()

print(f'Panel rows (all):  {len(panel)}')
print(f'Panel rows (clean): {len(panel_clean)}')
print(f'\nrole_type distribution:')
print(panel_clean['role_type'].value_counts())
print(f'\npartner_type distribution:')
print(panel_clean['partner_type'].value_counts())
panel_clean[['safehouse_id','month_start','partner_type','role_type',
             'avg_education_progress','avg_health_score',
             'process_recording_count','home_visitation_count',
             'tenure_days','occupancy_rate']].head()

## 4. Exploration

Before modeling, we visually inspect distributions and bivariate relationships. This mirrors the textbook's exploration phase (Ch. 6–8) and helps us identify potential non-linearities or outliers that could affect OLS assumptions.

In [ ]:
# ── Outcome distributions ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
panel_clean['avg_education_progress'].hist(bins=25, ax=axes[0], color='#42A5F5', edgecolor='white')
axes[0].set_title('Distribution: Avg Education Progress')
axes[0].set_xlabel('Score (0–100)')

panel_clean['avg_health_score'].hist(bins=25, ax=axes[1], color='#66BB6A', edgecolor='white')
axes[1].set_title('Distribution: Avg Health Score')
axes[1].set_xlabel('Score (1–5)')

plt.suptitle('Outcome Variable Distributions', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Role type & partner type vs outcomes ─────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.boxplot(data=panel_clean, x='role_type', y='avg_education_progress',
            palette='Set2', ax=axes[0,0])
axes[0,0].set_title('Role Type vs Education Progress')
axes[0,0].set_xlabel('')

sns.boxplot(data=panel_clean, x='role_type', y='avg_health_score',
            palette='Set2', ax=axes[0,1])
axes[0,1].set_title('Role Type vs Health Score')
axes[0,1].set_xlabel('')

sns.boxplot(data=panel_clean, x='partner_type', y='avg_education_progress',
            palette='Set3', ax=axes[1,0])
axes[1,0].set_title('Partner Type vs Education Progress')

sns.boxplot(data=panel_clean, x='partner_type', y='avg_health_score',
            palette='Set3', ax=axes[1,1])
axes[1,1].set_title('Partner Type vs Health Score')

plt.suptitle('Partner Characteristics vs Resident Outcomes', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Activity controls vs outcomes ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=panel_clean, x='process_recording_count',
                y='avg_education_progress', hue='role_type', alpha=0.6, ax=axes[0])
axes[0].set_title('Process Recordings vs Education Progress')

sns.scatterplot(data=panel_clean, x='home_visitation_count',
                y='avg_health_score', hue='role_type', alpha=0.6, ax=axes[1])
axes[1].set_title('Home Visitations vs Health Score')

plt.suptitle('Activity Intensity vs Outcomes (colored by Role Type)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Outcomes over time by safehouse ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sh_id, grp in panel_clean.groupby('safehouse_id'):
    grp_sorted = grp.sort_values('month_start')
    axes[0].plot(grp_sorted['month_start'], grp_sorted['avg_education_progress'],
                 alpha=0.7, label=f'SH{sh_id}')
    axes[1].plot(grp_sorted['month_start'], grp_sorted['avg_health_score'],
                 alpha=0.7, label=f'SH{sh_id}')

axes[0].set_title('Education Progress Over Time by Safehouse')
axes[0].set_ylabel('Avg Education Progress')
axes[0].legend(fontsize=8)

axes[1].set_title('Health Score Over Time by Safehouse')
axes[1].set_ylabel('Avg Health Score')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 5. Feature Engineering for Modeling

We dummy-encode categorical variables (`role_type`, `partner_type`, `program_area`) using `drop_first=True` to avoid perfect multicollinearity, as specified in the textbook (Ch. 9). The reference categories are the first alphabetically: `Evaluation` for role_type, `Individual` for partner_type.

Control variables included:
- `occupancy_rate` — safehouse crowding (a partner shouldn't be penalized for an overcrowded facility)
- `process_recording_count` — monthly service activity intensity 
- `home_visitation_count` — community engagement activity
- `tenure_days` — partner experience
- `month_num` — linear time trend (captures improvement over time unrelated to partner)

In [ ]:
# ── Dummy encoding ────────────────────────────────────────────────────
df_model = pd.get_dummies(
    panel_clean,
    columns=['role_type', 'partner_type', 'program_area'],
    drop_first=True,
    dtype=int
)

NUMERIC_CONTROLS = ['tenure_days', 'safehouse_count', 'occupancy_rate',
                    'process_recording_count', 'home_visitation_count', 'month_num']

DUMMY_COLS = [c for c in df_model.columns
              if any(c.startswith(p) for p in ['role_type_', 'partner_type_', 'program_area_'])]

FEATURES = NUMERIC_CONTROLS + DUMMY_COLS

X = df_model[FEATURES].astype(float)
y_edu    = df_model['avg_education_progress'].astype(float)
y_health = df_model['avg_health_score'].astype(float)

print('Feature matrix shape:', X.shape)
print('\nFeatures:')
for f in FEATURES:
    print(f'  {f}')

## 6a. MLR — Education Progress

We use statsmodels OLS (Ch. 9 approach) so we get full coefficient tables, p-values, and model fit statistics. The reference group for `role_type` is **Evaluation**, so all role coefficients are interpreted relative to an Evaluation-role partner.

In [ ]:
X_const = sm.add_constant(X)
ols_edu = sm.OLS(y_edu, X_const).fit()
print(ols_edu.summary())

In [ ]:
# ── Coefficient plot — partner-related features only ──────────────────
coef_table = ols_edu.summary2().tables[1].reset_index()
coef_table.columns = ['feature','coef','std_err','t','pval','ci_low','ci_high']

partner_coefs = coef_table[
    coef_table['feature'].str.contains('role_type|partner_type', regex=True)
].copy()
partner_coefs['significant'] = partner_coefs['pval'] < 0.05
partner_coefs['label'] = partner_coefs['feature'].str.replace('role_type_|partner_type_', '', regex=True)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#1976D2' if sig else '#BDBDBD' for sig in partner_coefs['significant']]
ax.barh(partner_coefs['label'], partner_coefs['coef'], color=colors)
ax.errorbar(
    partner_coefs['coef'], partner_coefs['label'],
    xerr=[partner_coefs['coef'] - partner_coefs['ci_low'],
          partner_coefs['ci_high'] - partner_coefs['coef']],
    fmt='none', color='black', capsize=4
)
ax.axvline(0, color='black', linewidth=0.8)
for _, row in partner_coefs.iterrows():
    ax.text(row['coef'] + 0.3, row['label'], f"p={row['pval']:.3f}", va='center', fontsize=8)
ax.set_xlabel('Coefficient vs. Reference (Evaluation role / Individual type)')
ax.set_title('MLR: Partner Feature Effects on Education Progress\n(blue = p < 0.05)')
plt.tight_layout()
plt.show()

print(f'R² = {ols_edu.rsquared:.3f} | Adj. R² = {ols_edu.rsquared_adj:.3f}')

## 6b. MLR — Health Score

The same model specification is applied to health score. Comparing results across both outcomes helps us identify whether certain partner traits are broadly beneficial or outcome-specific.

In [ ]:
ols_health = sm.OLS(y_health, X_const).fit()
print(ols_health.summary())
print(f'\nR² = {ols_health.rsquared:.3f} | Adj. R² = {ols_health.rsquared_adj:.3f}')

In [ ]:
coef_h = ols_health.summary2().tables[1].reset_index()
coef_h.columns = ['feature','coef','std_err','t','pval','ci_low','ci_high']

partner_h = coef_h[
    coef_h['feature'].str.contains('role_type|partner_type', regex=True)
].copy()
partner_h['significant'] = partner_h['pval'] < 0.05
partner_h['label'] = partner_h['feature'].str.replace('role_type_|partner_type_', '', regex=True)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#388E3C' if sig else '#BDBDBD' for sig in partner_h['significant']]
ax.barh(partner_h['label'], partner_h['coef'], color=colors)
ax.errorbar(
    partner_h['coef'], partner_h['label'],
    xerr=[partner_h['coef'] - partner_h['ci_low'],
          partner_h['ci_high'] - partner_h['coef']],
    fmt='none', color='black', capsize=4
)
ax.axvline(0, color='black', linewidth=0.8)
for _, row in partner_h.iterrows():
    ax.text(row['coef'] + 0.002, row['label'], f"p={row['pval']:.3f}", va='center', fontsize=8)
ax.set_xlabel('Coefficient vs. Reference (Evaluation role / Individual type)')
ax.set_title('MLR: Partner Feature Effects on Health Score\n(green = p < 0.05)')
plt.tight_layout()
plt.show()

## 7. OLS Assumption Diagnostics (Ch. 10)

Because our goal is **explanatory** (we want to trust the coefficients), we must verify the five core OLS assumptions before making claims. Per the textbook Ch. 10:

1. **Normality** — residuals should be approximately normal (check: histogram + Q-Q plot, Omnibus test)
2. **Multicollinearity** — features shouldn't be too linearly dependent (check: VIF; VIF > 10 = concern)
3. **Autocorrelation** — residuals shouldn't be correlated across time (check: Durbin-Watson ≈ 2 is ideal)
4. **Linearity** — residuals vs. fitted should show no pattern
5. **Homoscedasticity** — residual variance should be constant across fitted values

We run these for the education model (the primary target); findings apply qualitatively to health as well.

In [ ]:
# ── 1. Residual Normality ─────────────────────────────────────────────
residuals = ols_edu.resid

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(residuals, bins=30, density=True, alpha=0.7, color='#42A5F5', edgecolor='white')
axes[0].set_title(f'Residual Distribution\nOmnibus p = {ols_edu.test_normality("omnibus")[1]:.4f}')
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Density')

probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')

plt.suptitle('Assumption 1: Residual Normality', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Omnibus test statistic: {ols_edu.test_normality("omnibus")[0]:.3f}')
print(f'Omnibus p-value:        {ols_edu.test_normality("omnibus")[1]:.4f}')
print('Interpretation: p > 0.05 → residuals are approximately normal (assumption holds)')
print('                p < 0.05 → residuals deviate from normality (caution on p-values)')

In [ ]:
# ── 2. Multicollinearity — Variance Inflation Factor (VIF) ────────────
# VIF < 5: acceptable | 5-10: moderate | > 10: high concern (Ch. 10)

vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [
    variance_inflation_factor(X.values, i) for i in range(X.shape[1])
]
vif_data = vif_data.sort_values('VIF', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors_vif = ['#D32F2F' if v > 10 else '#FF8F00' if v > 5 else '#388E3C'
              for v in vif_data['VIF']]
ax.barh(vif_data['feature'][::-1], vif_data['VIF'][::-1], color=colors_vif[::-1])
ax.axvline(5,  color='orange', linestyle='--', linewidth=1.2, label='VIF = 5 (moderate)')
ax.axvline(10, color='red',    linestyle='--', linewidth=1.2, label='VIF = 10 (high)')
ax.set_xlabel('VIF')
ax.set_title('Assumption 2: Multicollinearity — VIF per Feature\n(green < 5, orange 5-10, red > 10)')
ax.legend()
plt.tight_layout()
plt.show()

high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print('HIGH VIF features (>10) — coefficient estimates may be unstable:')
    print(high_vif.to_string(index=False))
else:
    print('No features with VIF > 10. Multicollinearity is not a major concern.')
print(vif_data.to_string(index=False))

In [ ]:
# ── 3. Autocorrelation — Durbin-Watson ────────────────────────────────
# This matters because our panel is time-ordered within each safehouse.
# DW ≈ 2.0 means no autocorrelation; < 1.5 or > 2.5 is concerning.

from statsmodels.stats.stattools import durbin_watson

dw_stat = durbin_watson(residuals)
lag1_corr = pd.Series(residuals.values).autocorr(lag=1)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(residuals.values, alpha=0.6, linewidth=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'Assumption 3: Residuals in Row Order — Durbin-Watson = {dw_stat:.3f} | Lag-1 Autocorr = {lag1_corr:.3f}')
ax.set_xlabel('Observation (time-ordered within each safehouse)')
ax.set_ylabel('Residual')
plt.tight_layout()
plt.show()

print(f'Durbin-Watson statistic: {dw_stat:.3f}')
print(f'Lag-1 autocorrelation:   {lag1_corr:.3f}')
if 1.5 <= dw_stat <= 2.5:
    print('Interpretation: DW is within acceptable range. Autocorrelation is not a major concern.')
else:
    print('Interpretation: DW is outside 1.5-2.5. Some autocorrelation may be present — treat p-values with caution.')

In [ ]:
# ── 4 & 5. Linearity + Homoscedasticity — Residuals vs. Fitted ────────
fitted = ols_edu.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(fitted, residuals, alpha=0.4, s=20, color='#5C6BC0')
axes[0].axhline(0, color='black', linewidth=0.8)
# Lowess smoother to detect patterns
from statsmodels.nonparametric.smoothers_lowess import lowess
lo = lowess(residuals, fitted, frac=0.4)
axes[0].plot(lo[:, 0], lo[:, 1], color='red', linewidth=1.5, label='LOWESS')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Assumptions 4 & 5: Residuals vs. Fitted\n(red line should be flat near 0 if assumptions hold)')
axes[0].legend()

axes[1].scatter(fitted, np.sqrt(np.abs(residuals)), alpha=0.4, s=20, color='#EF5350')
lo2 = lowess(np.sqrt(np.abs(residuals)), fitted, frac=0.4)
axes[1].plot(lo2[:, 0], lo2[:, 1], color='black', linewidth=1.5)
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('√|Residuals|')
axes[1].set_title('Scale-Location Plot\n(flat trend = homoscedastic variance)')

plt.tight_layout()
plt.show()

## 8. Decision Tree — Alternative Approach

Decision trees (Ch. 12) capture non-linear relationships and feature interactions that OLS cannot. We use a shallow tree (`max_depth=4`) to keep it interpretable. The **feature importance** plot shows which variables produce the greatest reduction in prediction error — complementing the OLS coefficients.

In [ ]:
tree_edu = DecisionTreeRegressor(max_depth=4, min_samples_leaf=8, random_state=42)
tree_edu.fit(X, y_edu)

importance_df = (
    pd.DataFrame({'feature': X.columns, 'importance': tree_edu.feature_importances_})
    .sort_values('importance', ascending=False)
    .head(12)
)

fig, ax = plt.subplots(figsize=(10, 6))
colors_imp = [
    '#FF7043' if ('role_type' in f or 'partner_type' in f) else '#78909C'
    for f in importance_df['feature']
]
ax.barh(importance_df['feature'][::-1], importance_df['importance'][::-1],
        color=colors_imp[::-1])
ax.set_xlabel('Feature Importance (impurity reduction)')
ax.set_title('Decision Tree: Feature Importances — Education Progress\n(orange = partner characteristic features)')
plt.tight_layout()
plt.show()

print('Top 5 split rules (tree depth 1-2):')
print(export_text(tree_edu, feature_names=list(X.columns), max_depth=2))

In [ ]:
# ── Decision tree for health score ───────────────────────────────────
tree_health = DecisionTreeRegressor(max_depth=4, min_samples_leaf=8, random_state=42)
tree_health.fit(X, y_health)

imp_h = (
    pd.DataFrame({'feature': X.columns, 'importance': tree_health.feature_importances_})
    .sort_values('importance', ascending=False)
    .head(12)
)

fig, ax = plt.subplots(figsize=(10, 6))
colors_h = [
    '#FF7043' if ('role_type' in f or 'partner_type' in f) else '#78909C'
    for f in imp_h['feature']
]
ax.barh(imp_h['feature'][::-1], imp_h['importance'][::-1], color=colors_h[::-1])
ax.set_xlabel('Feature Importance (impurity reduction)')
ax.set_title('Decision Tree: Feature Importances — Health Score\n(orange = partner characteristic features)')
plt.tight_layout()
plt.show()

## 9. Cross-Validated Model Comparison (Ch. 15)

**Why cross-validation matters here:** The textbook (Ch. 15) is explicit that a single train/test split produces a single estimate of performance that can change depending on which samples land in the test set — *especially problematic for small/medium datasets*. We use two strategies:

- **Leave-One-Out CV (LOOCV)** — recommended for datasets fewer than 50 unique entities (we have 9 safehouses). Each fold holds out one observation at a time.
- **Repeated K-Fold (5-fold, 3 repeats)** — gives a more stable estimate by averaging across multiple random splits.

We compare MLR vs. Decision Tree (depth=4 and depth=2) on both outcomes.

In [ ]:
loocv  = LeaveOneOut()
rkf    = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

model_specs = {
    'MLR':                  LinearRegression(),
    'Tree (depth=4)':       DecisionTreeRegressor(max_depth=4, min_samples_leaf=8, random_state=42),
    'Tree (depth=2)':       DecisionTreeRegressor(max_depth=2, min_samples_leaf=8, random_state=42),
}

results = []
for outcome_name, y_target in [('Education Progress', y_edu), ('Health Score', y_health)]:
    for model_name, model in model_specs.items():
        # Repeated K-Fold
        rkf_mae = -cross_val_score(model, X, y_target, cv=rkf,
                                   scoring='neg_mean_absolute_error')
        rkf_r2  =  cross_val_score(model, X, y_target, cv=rkf, scoring='r2')
        # LOOCV
        loo_mae = -cross_val_score(model, X, y_target, cv=loocv,
                                   scoring='neg_mean_absolute_error')
        results.append({
            'Outcome':       outcome_name,
            'Model':         model_name,
            'RKF MAE mean':  round(rkf_mae.mean(), 3),
            'RKF MAE std':   round(rkf_mae.std(), 3),
            'RKF R² mean':   round(rkf_r2.mean(), 3),
            'RKF R² std':    round(rkf_r2.std(), 3),
            'LOOCV MAE mean':round(loo_mae.mean(), 3),
        })

cv_df = pd.DataFrame(results)
print('=== Cross-Validation Results ===')
print(cv_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for row_idx, outcome in enumerate(['Education Progress', 'Health Score']):
    sub = cv_df[cv_df['Outcome'] == outcome]
    palette = sns.color_palette('muted', 3)

    # MAE
    axes[row_idx, 0].bar(sub['Model'], sub['RKF MAE mean'],
                         yerr=sub['RKF MAE std'], capsize=5, color=palette)
    axes[row_idx, 0].set_title(f'{outcome}: Repeated K-Fold MAE (lower = better)')
    axes[row_idx, 0].set_ylabel('MAE')
    axes[row_idx, 0].tick_params(axis='x', rotation=15)

    # R²
    axes[row_idx, 1].bar(sub['Model'], sub['RKF R² mean'],
                         yerr=sub['RKF R² std'], capsize=5, color=palette)
    axes[row_idx, 1].set_title(f'{outcome}: Repeated K-Fold R² (higher = better)')
    axes[row_idx, 1].set_ylabel('R²')
    axes[row_idx, 1].tick_params(axis='x', rotation=15)

plt.suptitle('Model Comparison: Repeated K-Fold Cross-Validation (5 splits × 3 repeats)', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Results Summary & Business Recommendations

This section synthesizes all findings into plain-language insights for Lighthouse leadership.

In [ ]:
# ── Best model per outcome ────────────────────────────────────────────
print('=== BEST MODEL PER OUTCOME (by Repeated K-Fold R²) ===')
for outcome in ['Education Progress', 'Health Score']:
    sub = cv_df[cv_df['Outcome'] == outcome]
    best = sub.loc[sub['RKF R² mean'].idxmax()]
    print(f'  {outcome}: {best["Model"]}  |  R² = {best["RKF R² mean"]}  |  MAE = {best["RKF MAE mean"]}')

print()
print('=== SIGNIFICANT PARTNER EFFECTS (Education Progress, MLR) ===')
sig_edu = partner_coefs[partner_coefs['pval'] < 0.05]
if len(sig_edu) > 0:
    for _, r in sig_edu.iterrows():
        direction = 'higher' if r['coef'] > 0 else 'lower'
        print(f"  {r['label']}: {abs(r['coef']):.2f} pts {direction} progress vs. reference (p={r['pval']:.3f})")
else:
    print('  No features reached p < 0.05 in education model.')
    print('  Review directional coefficients and feature importances for guidance.')

print()
print('=== SIGNIFICANT PARTNER EFFECTS (Health Score, MLR) ===')
sig_health = partner_h[partner_h['pval'] < 0.05]
if len(sig_health) > 0:
    for _, r in sig_health.iterrows():
        direction = 'higher' if r['coef'] > 0 else 'lower'
        print(f"  {r['label']}: {abs(r['coef']):.3f} pts {direction} health score vs. reference (p={r['pval']:.3f})")
else:
    print('  No features reached p < 0.05 in health model.')

## 11. Limitations & Next Steps

**Assignment Bias (Confounding):** If leadership deliberately assigns experienced partners to struggling safehouses, the model may show experienced partners associated with *lower* scores — not because they're worse, but because they're sent to harder situations. This is an observational dataset with no randomized assignment, so coefficients should be interpreted as correlations, not strict causation.

**Panel Clustering:** Because multiple months come from the same safehouse, observations are not fully independent. A fixed-effects panel model (e.g., `safehouse_id` dummies or `statsmodels.PanelOLS`) would more rigorously account for unobserved safehouse-level differences. This is a recommended next step.

**Sample Size:** With 9 safehouses and 4 role types, the dataset is still relatively small for drawing confident causal conclusions. Collecting more months of data will strengthen any future analysis.

**Next Steps:**
1. Implement a fixed-effects panel model to control for safehouse-level heterogeneity
2. Re-run analysis as additional monthly metrics are collected
3. Explore interaction terms (e.g., does `role_type × tenure` matter?)
4. Use model coefficients to populate the Admin Dashboard's "Partner Insights" widget, updated quarterly